In [0]:
from pyspark.sql import functions as F

PREFIX = "workspace.gep_public_procurement"

silver = spark.table(f"{PREFIX}.silver_precos")

assert silver.count() > 0, "A Silver está vazia."

gold = (
    silver
    .withColumn("mes", F.trunc("dataResultado", "month"))
    .groupBy(
        "codigoItemCatalogo",
        "nomeUnidadeFornecimento",
        "mes",
        "unidade_canonica"
    )
    .agg(
        F.sum(
            F.col("canonical_quantity") * F.col("canonical_price")
        ).alias("_valor_total"),

        F.sum("canonical_quantity").alias(
            "quantidade_canonica_total"
        ),

        F.count("*").alias("qtd_transacoes"),

        F.countDistinct("niFornecedor").alias("qtd_fornecedores")
    )
    .withColumn(
        "valor_medio",
        F.col("_valor_total") / F.col("quantidade_canonica_total")
    )
    .select(
        "codigoItemCatalogo",
        "nomeUnidadeFornecimento",
        "mes",
        "unidade_canonica",
        "valor_medio",
        "qtd_transacoes",
        "quantidade_canonica_total",
        "qtd_fornecedores"
    )
)

# A chave exigida deve identificar uma única linha.
duplicidades_gold = (
    gold
    .groupBy(
        "codigoItemCatalogo",
        "nomeUnidadeFornecimento",
        "mes"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicidades_gold == 0, "Existem chaves duplicadas na Gold."

(
    gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{PREFIX}.gold_precos_mensais")
)

print(f"Tabela salva: {PREFIX}.gold_precos_mensais")
print(f"Registros na Gold: {gold.count()}")

display(
    gold.orderBy(
        "codigoItemCatalogo",
        "nomeUnidadeFornecimento",
        "mes"
    ).limit(20)
)

Tabela salva: workspace.gep_public_procurement.gold_precos_mensais
Registros na Gold: 292


codigoItemCatalogo,nomeUnidadeFornecimento,mes,unidade_canonica,valor_medio,qtd_transacoes,quantidade_canonica_total,qtd_fornecedores
217129,PACOTE,2025-07-01,KG,9.899999999999999,1,1318.0,1
217129,PACOTE,2025-08-01,KG,14.021426107949646,10,4639.2,10
217129,PACOTE,2025-09-01,KG,19.549944883197675,5,3311.15,4
217129,PACOTE,2025-10-01,KG,7.564813105538649,7,10075.740000000002,5
217129,PACOTE,2025-11-01,KG,9.767105263157893,2,760.0,2
217129,PACOTE,2025-12-01,KG,9.125,1,14100.0,1
217129,PACOTE,2026-01-01,KG,11.095839636913768,6,1322.0,6
217129,PACOTE,2026-02-01,KG,8.929695612668743,6,6163.200000000001,5
217129,PACOTE,2026-03-01,KG,8.896113547758285,4,1641.6000000000001,4
217129,PACOTE,2026-04-01,KG,9.28734756097561,4,2072.96,4


In [0]:
consultas = {
    "silver_quantidades_precos_invalidos": f"""
        SELECT COUNT(*) AS ocorrencias
        FROM {PREFIX}.silver_precos
        WHERE canonical_quantity IS NULL
           OR canonical_quantity <= 0
           OR isnan(canonical_quantity)
           OR ABS(canonical_quantity) = CAST('Infinity' AS DOUBLE)
           OR canonical_price IS NULL
           OR canonical_price <= 0
           OR isnan(canonical_price)
           OR ABS(canonical_price) = CAST('Infinity' AS DOUBLE)
    """,

    "silver_codigos_fora_catalogo": f"""
        SELECT COUNT(*) AS ocorrencias
        FROM {PREFIX}.silver_precos s
        LEFT ANTI JOIN {PREFIX}.ref_catalogo c
          ON s.codigoItemCatalogo = TRIM(c.codigoItemCatalogo)
    """,

    "gold_chaves_duplicadas": f"""
        SELECT COUNT(*) AS ocorrencias
        FROM (
            SELECT
                codigoItemCatalogo,
                nomeUnidadeFornecimento,
                mes
            FROM {PREFIX}.gold_precos_mensais
            GROUP BY
                codigoItemCatalogo,
                nomeUnidadeFornecimento,
                mes
            HAVING COUNT(*) > 1
        ) duplicadas
    """,

    "gold_valores_invalidos": f"""
        SELECT COUNT(*) AS ocorrencias
        FROM {PREFIX}.gold_precos_mensais
        WHERE mes IS NULL
           OR DAYOFMONTH(mes) <> 1
           OR valor_medio IS NULL
           OR valor_medio <= 0
           OR isnan(valor_medio)
           OR ABS(valor_medio) = CAST('Infinity' AS DOUBLE)
           OR quantidade_canonica_total IS NULL
           OR quantidade_canonica_total <= 0
           OR isnan(quantidade_canonica_total)
           OR ABS(quantidade_canonica_total) = CAST('Infinity' AS DOUBLE)
           OR qtd_transacoes <= 0
           OR qtd_fornecedores > qtd_transacoes
    """
}

resultados = []

for nome, consulta in consultas.items():
    ocorrencias = int(spark.sql(consulta).first()["ocorrencias"])
    resultados.append((
        nome,
        ocorrencias,
        "OK" if ocorrencias == 0 else "FALHOU"
    ))
    print(f"{nome}: {ocorrencias} ocorrência(s)")

# Confirma que todas as transações da Silver entraram na Gold.
n_silver = silver.count()

n_transacoes_gold = (
    spark.table(f"{PREFIX}.gold_precos_mensais")
    .agg(F.sum("qtd_transacoes").alias("total"))
    .first()["total"]
)

diferenca = abs(n_silver - int(n_transacoes_gold or 0))

resultados.append((
    "reconciliacao_transacoes_silver_gold",
    diferenca,
    "OK" if diferenca == 0 else "FALHOU"
))

validacoes = spark.createDataFrame(
    resultados,
    "validacao STRING, ocorrencias LONG, resultado STRING"
)

(
    validacoes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{PREFIX}.validacoes_pipeline")
)

display(validacoes)

assert all(
    ocorrencias == 0
    for _, ocorrencias, _ in resultados
), "Uma ou mais validações falharam. Confira os resultados."

print("Todas as validações passaram.")

silver_quantidades_precos_invalidos: 0 ocorrência(s)
silver_codigos_fora_catalogo: 0 ocorrência(s)
gold_chaves_duplicadas: 0 ocorrência(s)
gold_valores_invalidos: 0 ocorrência(s)


validacao,ocorrencias,resultado
silver_quantidades_precos_invalidos,0,OK
silver_codigos_fora_catalogo,0,OK
gold_chaves_duplicadas,0,OK
gold_valores_invalidos,0,OK
reconciliacao_transacoes_silver_gold,0,OK


Todas as validações passaram.


In [0]:
bronze = spark.table(f"{PREFIX}.bronze_precos")
gold_salva = spark.table(f"{PREFIX}.gold_precos_mensais")
auditoria = spark.table(f"{PREFIX}.auditoria_silver")

runs = bronze.select("run_id").distinct().collect()
assert len(runs) == 1, "A Bronze contém mais de uma execução."
run_id = runs[0]["run_id"]

logs = (
    spark.table(f"{PREFIX}.log_extracao")
    .filter(F.col("run_id") == run_id)
)

resumo = spark.createDataFrame(
    [
        ("Bronze", bronze.count()),
        ("Silver", silver.count()),
        ("Gold", gold_salva.count())
    ],
    "camada STRING, quantidade_registros LONG"
)

print(f"Execução: {run_id}")

print("\nCONTAGENS POR CAMADA")
display(resumo)

print("\nAUDITORIA DAS TRANSFORMAÇÕES")
display(auditoria.orderBy("ordem"))

print("\nRESULTADO DAS CONSULTAS À API")
display(
    logs.groupBy("status_http", "resultado_consulta")
    .agg(
        F.count("*").alias("quantidade_consultas"),
        F.sum("qtd_registros").alias("registros_recebidos")
    )
)

print("\nTRANSAÇÕES SEM IDENTIFICAÇÃO DE FORNECEDOR")
print(silver.filter(F.col("niFornecedor").isNull()).count())

print("\nAMOSTRA DA GOLD — ATÉ 20 LINHAS")
display(
    gold_salva.orderBy(
        "codigoItemCatalogo",
        "nomeUnidadeFornecimento",
        "mes"
    ).limit(20)
)

if gold_salva.count() < 10:
    print(
        "ATENÇÃO: a Gold possui menos de 10 linhas. "
        "Precisamos conferir isso antes de finalizar a entrega."
    )

Execução: a7ff4fb4-9ed6-485d-a4d0-3611b361f67b

CONTAGENS POR CAMADA


camada,quantidade_registros
Bronze,2000
Silver,1439
Gold,292



AUDITORIA DAS TRANSFORMAÇÕES


run_id,ordem,etapa,linhas_entrada,linhas_saida,linhas_removidas
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,1,correspondencia_catalogo,2000,1461,539
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,2,tipos_valores_validos,1461,1461,0
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,3,duplicidades_exatas,1461,1461,0
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,4,unidades_compativeis,1461,1439,22
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,5,conversao_canonica,1439,1439,0



RESULTADO DAS CONSULTAS À API


status_http,resultado_consulta,quantidade_consultas,registros_recebidos
200,sucesso,20,2000



TRANSAÇÕES SEM IDENTIFICAÇÃO DE FORNECEDOR
0

AMOSTRA DA GOLD — ATÉ 20 LINHAS


codigoItemCatalogo,nomeUnidadeFornecimento,mes,unidade_canonica,valor_medio,qtd_transacoes,quantidade_canonica_total,qtd_fornecedores
217129,PACOTE,2025-07-01,KG,9.899999999999999,1,1318.0,1
217129,PACOTE,2025-08-01,KG,14.021426107949646,10,4639.2,10
217129,PACOTE,2025-09-01,KG,19.549944883197675,5,3311.15,4
217129,PACOTE,2025-10-01,KG,7.56481310553865,7,10075.74,5
217129,PACOTE,2025-11-01,KG,9.767105263157893,2,760.0,2
217129,PACOTE,2025-12-01,KG,9.125,1,14100.0,1
217129,PACOTE,2026-01-01,KG,11.095839636913768,6,1322.0,6
217129,PACOTE,2026-02-01,KG,8.929695612668743,6,6163.200000000001,5
217129,PACOTE,2026-03-01,KG,8.896113547758285,4,1641.6000000000001,4
217129,PACOTE,2026-04-01,KG,9.28734756097561,4,2072.96,4
